In [5]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

response = client.chat.completions.create(
    model="qwen2.5:3b",
    messages=[
        {"role": "system", "content": "You are a helpful travel assistant."},
        {"role": "user", "content": "Suggest a hill station near Bengaluru."}
    ],
    max_tokens=50
)

print(response.choices[0].message.content)

A great hill station near Bengaluru is Mudumalai. Here are some reasons why Mudumalai can be a fantastic choice:

1. **Location**: Mudumalai is around 130 km northeast of Bengaluru. It


# Catchy description lines for 50k products - which you can assign to the LLM - give an id - i will fetch the result later - batch processing - 24h - 20-30%

# In which format we create the batch file - jsonl


# There 3 steps
1. Bundle the request - Many requests together in one file - jsonl
2. Uploading and queue - Process these whenever you can
3. Off peak Execution - Within 24 Hours




In [7]:
# Create the client

import os
import json
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [9]:
# Step 1 - Prepare teh file
products = ["Steel water bottle 1L", "Cotton bedsheet queen size", "Wireless mouse"]

# 'batch_request.jsonl' - write mode - run the loop over this products list -

with open('batch_request.jsonl', 'w') as f:
  for i, product in enumerate(products, start = 1):
    request = {
        'custom_id':f"product-{i}",           # Our own label to matn the result later
        'method' : 'POST',
        'url': '/v1/chat/completions',
        'body':{
            'model':'gpt-4o-mini',
            'messages':[
                {'role':'system','content':'Write a 30-word product description for an online store.'},
                {'role':'user', 'content':product}
            ],
            'max_completion_tokens' : 80,
        }
    }
  f.write(json.dumps(request) + '\n')



In [12]:

import json
import httpx
from openai import OpenAI

# Connect to your local Ollama server
http_client = httpx.Client(
    trust_env=False,
    timeout=httpx.Timeout(180.0, connect=10.0)
)

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
    http_client=http_client,
    max_retries=0
)

input_file = "batch_request.jsonl"
output_file = "batch_response.jsonl"

# Read requests and process them one by one
try:
    with open(input_file, "r", encoding="utf-8") as infile, \
         open(output_file, "w", encoding="utf-8") as outfile:

        for line_number, line in enumerate(infile, start=1):
            if not line.strip():
                continue

            request = json.loads(line)
            custom_id = request.get("custom_id", f"request-{line_number}")
            body = request["body"]

            try:
                response = client.chat.completions.create(
                    model="qwen2.5:3b",
                    messages=body["messages"],
                    max_tokens=body.get("max_completion_tokens", 80)
                )

                result = {
                    "custom_id": custom_id,
                    "response": response.choices[0].message.content,
                    "error": None
                }

            except Exception as e:
                result = {
                    "custom_id": custom_id,
                    "response": None,
                    "error": str(e)
                }

            # Save one result per line
            outfile.write(json.dumps(result, ensure_ascii=False) + "\n")
            outfile.flush()

            print(f"Processed: {custom_id}")

    print(f"\nBatch processing finished. Results saved to {output_file}")

finally:
    http_client.close()


Processed: product-3

Batch processing finished. Results saved to batch_response.jsonl


In [13]:
import json

with open("batch_response.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        result = json.loads(line)
        print(f"\n{result['custom_id']}:")
        print(result["response"] or result["error"])


product-3:
Ergonomic wireless mouse for precision typing and gaming. Ultra-fast, long-range connectivity - perfect for your next tech adventure.


In [14]:

import json
from pathlib import Path

import httpx
from openai import OpenAI

# Local files
INPUT_FILE = Path("batch_request.jsonl")
OUTPUT_FILE = Path("batch_response.jsonl")

# Local Ollama model
MODEL = "qwen2.5:3b"

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"{INPUT_FILE} not found. Check your notebook's working directory."
    )

# Connect to Ollama, not OpenAI's cloud API
http_client = httpx.Client(
    trust_env=False,
    timeout=httpx.Timeout(180.0, connect=10.0)
)

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
    http_client=http_client,
    max_retries=0
)

processed = 0
failed = 0

try:
    with INPUT_FILE.open("r", encoding="utf-8") as infile, \
         OUTPUT_FILE.open("w", encoding="utf-8") as outfile:

        for line_number, line in enumerate(infile, start=1):
            if not line.strip():
                continue

            request = json.loads(line)
            custom_id = request.get(
                "custom_id", f"request-{line_number}"
            )

            try:
                body = request["body"]

                # Execute this request using the local model
                response = client.chat.completions.create(
                    model=MODEL,
                    messages=body["messages"],
                    max_tokens=body.get(
                        "max_tokens",
                        body.get("max_completion_tokens", 80)
                    )
                )

                # Keep the response associated with its original ID
                result = {
                    "custom_id": custom_id,
                    "response": {
                        "status_code": 200,
                        "body": response.model_dump(mode="json")
                    },
                    "error": None
                }

                processed += 1

                print(f"Completed: {custom_id}")
                print(response.choices[0].message.content)
                print("-" * 50)

            except Exception as exc:
                result = {
                    "custom_id": custom_id,
                    "response": None,
                    "error": {
                        "message": str(exc)
                    }
                }

                failed += 1
                print(f"Failed: {custom_id} — {exc}")

            # Write one result per JSONL line
            outfile.write(
                json.dumps(result, ensure_ascii=False) + "\n"
            )
            outfile.flush()

    print("\nLocal batch processing finished.")
    print(f"Successful requests: {processed}")
    print(f"Failed requests: {failed}")
    print(f"Output file: {OUTPUT_FILE.resolve()}")

finally:
    http_client.close()


Completed: product-3
Enhance your digital experience with the sleek, wireless mouse - perfect for speed and precision on your PC or Mac.
--------------------------------------------------

Local batch processing finished.
Successful requests: 1
Failed requests: 0
Output file: C:\Users\ankit\Desktop\LearnBay\Learnbay-Learning\GenAI\Codes\batch_response.jsonl


In [15]:

import json

with open("batch_response.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        item = json.loads(line)

        print(f"\nID: {item['custom_id']}")

        if item["response"]:
            content = item["response"]["body"]["choices"][0][
                "message"
            ]["content"]
            print("Description:", content)
        else:
            print("Error:", item["error"]["message"])



ID: product-3
Description: Enhance your digital experience with the sleek, wireless mouse - perfect for speed and precision on your PC or Mac.


In [16]:
# Step 3 - Checking and collecting the results

import json

# Read the output file generated by local batch processing
with open("batch_response.jsonl", "r", encoding="utf-8") as file:
    results = file.read()

print(results)
print(type(results))

{"custom_id": "product-3", "response": {"status_code": 200, "body": {"id": "chatcmpl-984", "choices": [{"finish_reason": "stop", "index": 0, "logprobs": null, "message": {"content": "Enhance your digital experience with the sleek, wireless mouse - perfect for speed and precision on your PC or Mac.", "refusal": null, "role": "assistant", "annotations": null, "audio": null, "function_call": null, "tool_calls": null}}], "created": 1792552921, "model": "qwen2.5:3b", "object": "chat.completion", "metadata": null, "moderation": null, "service_tier": null, "system_fingerprint": "fp_ollama", "usage": {"completion_tokens": 24, "prompt_tokens": 29, "total_tokens": 53, "completion_tokens_details": null, "prompt_tokens_details": {"audio_tokens": null, "cache_write_tokens": null, "cached_tokens": 0, "image_tokens": null, "text_tokens": null}}}}, "error": null}

<class 'str'>
